# Simple Image Classifier

Yeh notebook aapko ek pre-trained neural network use karke images ko classify karna sikhati hai.

**Aap kya seekhenge:**
- Pre-trained model ko load aur use kaise karein
- Image preprocessing
- Image par prediction karna
- Confidence scores ko samajhna

**Use case:** Images mein objects ko identify karna (jaise "cat", "dog", "car" etc.)

---

## Step 1: Zaroori Libraries Import Karein

Chalo woh tools import karte hain jinki hume zaroorat hai. Agar inmein se kuch aapko abhi poori tarah samajh nahi aate toh chinta mat kijiye!

In [ ]:
# Core libraries
import numpy as np
from PIL import Image
import requests
from io import BytesIO

# TensorFlow for deep learning
try:
    import tensorflow as tf
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
    print("✅ TensorFlow loaded successfully!")
    print(f"   Version: {tf.__version__}")
except ImportError:
    print("❌ Please install TensorFlow: pip install tensorflow")

## Step 2: Pre-trained Model Load Karein

Hum **MobileNetV2** use karenge, jo pehle se hi lakhon images par trained ek neural network hai.

Ise **Transfer Learning** kehte hain - kisi aur ke trained model ko use karna!

In [ ]:
print("📦 Loading pre-trained MobileNetV2 model...")
print("   This may take a minute on first run (downloading weights)...")

# Load the model
# include_top=True means we use the classification layer
# weights='imagenet' means it was trained on ImageNet dataset
model = MobileNetV2(weights='imagenet', include_top=True)

print("✅ Model loaded!")
print(f"   The model can recognize 1000 different object categories")

## Step 3: Helper Functions

Chalo apne model ke liye images load aur prepare karne ke functions banate hain.

In [ ]:
def load_image_from_url(url):
    """
    Load an image from a URL.
    
    Args:
        url: Web address of the image
        
    Returns:
        PIL Image object
    """
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img


def prepare_image(img):
    """
    Prepare an image for the model.
    
    Steps:
    1. Resize to 224x224 (model's expected size)
    2. Convert to array
    3. Add batch dimension
    4. Preprocess for MobileNetV2
    
    Args:
        img: PIL Image
        
    Returns:
        Preprocessed image array
    """
    # Resize to 224x224 pixels
    img = img.resize((224, 224))
    
    # Convert to numpy array
    img_array = np.array(img)
    
    # Add batch dimension (model expects multiple images)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Preprocess for MobileNetV2
    img_array = preprocess_input(img_array)
    
    return img_array


def classify_image(img):
    """
    Classify an image and return top predictions.
    
    Args:
        img: PIL Image
        
    Returns:
        List of (class_name, confidence) tuples
    """
    # Prepare the image
    img_array = prepare_image(img)
    
    # Make prediction
    predictions = model.predict(img_array, verbose=0)
    
    # Decode predictions to human-readable labels
    # top=5 means we get the top 5 most likely classes
    decoded = decode_predictions(predictions, top=5)[0]
    
    # Convert to simpler format
    results = [(label, float(confidence)) for (_, label, confidence) in decoded]
    
    return results


print("✅ Helper functions ready!")

## Step 4: Sample Images par Test Karein

Chalo internet se kuch images ko classify karne ki koshish karte hain!

In [ ]:
# Sample images to classify
# These are from Unsplash (free stock photos)
test_images = [
    {
        "url": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=400",
        "description": "A cat"
    },
    {
        "url": "https://images.unsplash.com/photo-1552053831-71594a27632d?w=400",
        "description": "A dog"
    },
    {
        "url": "https://images.unsplash.com/photo-1511919884226-fd3cad34687c?w=400",
        "description": "A car"
    },
]

print(f"🧪 Testing on {len(test_images)} images...")
print("=" * 70)

### Har Image ko Classify Karein

In [ ]:
for i, img_data in enumerate(test_images, 1):
    print(f"\n📸 Image {i}: {img_data['description']}")
    print("-" * 70)
    
    try:
        # Load image
        img = load_image_from_url(img_data['url'])
        
        # Display image
        display(img.resize((200, 200)))  # Show smaller version
        
        # Classify
        results = classify_image(img)
        
        # Show predictions
        print("\n🎯 Top 5 Predictions:")
        for rank, (label, confidence) in enumerate(results, 1):
            # Create a visual bar
            bar_length = int(confidence * 50)
            bar = "█" * bar_length
            
            print(f"  {rank}. {label:20s} {confidence*100:5.2f}% {bar}")
        
    except Exception as e:
        print(f"❌ Error: {e}")

print("\n" + "=" * 70)

## Step 5: Apni Khud ki Images Try Karein!

Neeche diye gaye URL ko kisi bhi image URL se replace kar dein jise aap classify karna chahte hain.

In [ ]:
# Try your own image!
# Replace this URL with any image URL
custom_image_url = "https://images.unsplash.com/photo-1472491235688-bdc81a63246e?w=400"  # A flower

print("🖼️  Classifying your custom image...")
print("=" * 70)

try:
    # Load and show image
    img = load_image_from_url(custom_image_url)
    display(img.resize((300, 300)))
    
    # Classify
    results = classify_image(img)
    
    # Show results
    print("\n🎯 Top 5 Predictions:")
    print("-" * 70)
    for rank, (label, confidence) in enumerate(results, 1):
        bar_length = int(confidence * 50)
        bar = "█" * bar_length
        print(f"  {rank}. {label:20s} {confidence*100:5.2f}% {bar}")
    
    # Highlight top prediction
    top_label, top_confidence = results[0]
    print("\n" + "=" * 70)
    print(f"\n🏆 Best guess: {top_label} ({top_confidence*100:.2f}% confident)")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("   Make sure the URL points to a valid image!")

## 💡 Abhi Kya Hua?

1. **Humne ek pre-trained model load kiya** - MobileNetV2 ko lakhon pictures par train kiya gaya tha
2. **Humne pictures ko preprocess kiya** - unhe model ke liye sahi size aur format mein badla
3. **Model ne prediction kiya** - yeh 1000 object classes ke liye probabilities batata hai
4. **Humne results ko decode kiya** - numbers ko insaanon ke padhne laayak labels mein badla

### Confidence Scores ko Samajhna

- **90-100%**: Bahut zyada bharosemand (lagbhag pakka sahi)
- **70-90%**: Bharosemand (shayad sahi)
- **50-70%**: Thoda bharosemand (sahi ho sakta hai)
- **50% se kam**: Zyada bharosemand nahi (uncertain)

### Prediction galat kyun ho sakti hai?

- **Ajeeb angle ya lighting** - model ko normal pictures par train kiya gaya tha
- **Kai objects** - model ek main object ki ummeed karta hai
- **Rare objects** - model sirf 1000 categories jaanta hai
- **Low quality picture** - blurry ya pixelated pictures ko pehchaanna mushkil hota hai

---

## 🚀 Next Steps

1. **Alag-alag pictures try karein:**
   - [Unsplash](https://unsplash.com) par pictures dhoondhein
   - Right-click karein → "Copy image address" se URL paayein

2. **Experiment karein:**
   - Abstract art ke saath kya hota hai?
   - Kya yeh alag-alag angles se objects ko pehchaan sakta hai?
   - Yeh kai objects ko kaise handle karta hai?

3. **Aur seekhein:**
   - [Computer Vision lessons](../lessons/4-ComputerVision/README.md) explore karein
   - Apna khud ka image classifier train karna seekhein
   - Samjhein ki CNNs (Convolutional Neural Networks) kaise kaam karte hain

---

## 🎉 Congratulations!

Aapne abhi-abhi ek state-of-the-art neural network use karke ek image classifier banaya hai!

Yahi technology power deti hai:
- Google Photos (aapki pictures ko organize karna)
- Self-driving cars (objects ko pehchaanna)
- Medical diagnosis (X-rays ka analysis karna)
- Quality control (defects ka pata lagana)

Explore karte rahein aur seekhte rahein! 🚀

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->